# AI Cycling Coach — JupyterHub Training (RTX A6000)

**Just click Run All.** Handles clone, deps, data generation, and training automatically.

Only change **Cell 1** if you want the repo in a different folder.

In [ ]:
# ── CONFIG — only cell you might need to edit ─────────────────────────────────
import os

REPO     = 'https://github.com/yossibello/ai-coach.git'
WORKDIR  = os.path.expanduser('~/ai-coach')   # repo lives here
ATHLETES = 50_000                             # athletes to generate (~8 min on A6000)

print(f'WORKDIR  : {WORKDIR}')
print(f'REPO     : {REPO}')
print(f'ATHLETES : {ATHLETES:,}')


In [ ]:
# ── 1. Clone / update repo ────────────────────────────────────────────────────
import os, subprocess

WORKDIR = os.path.expanduser('~/ai-coach')   # re-state so cell works standalone

if not os.path.exists(WORKDIR):
    print('Cloning repo…')
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)
else:
    print('Repo already exists — pulling latest…')
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)

print(f'✓ Repo ready at {WORKDIR}')

In [ ]:
# ── 2. Check GPU ──────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    device  = 'cuda'
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {vram_gb} GB')
    print(f'CUDA    : {torch.version.cuda}')
elif torch.backends.mps.is_available():
    device  = 'mps'
    vram_gb = 16
    print('Apple Silicon MPS')
else:
    device  = 'cpu'
    vram_gb = 0
    print('⚠  No GPU found — CPU only, training will be very slow')

print(f'PyTorch : {torch.__version__}  |  device: {device}')

In [ ]:
# ── 3. Install missing Python packages ────────────────────────────────────────
# PyTorch is NOT auto-installed here — on JupyterHub it is usually pre-installed
# system-wide or in a conda env. If cell 2 raised ImportError, see instructions
# at the bottom of this cell.
#
# Everything else (pandas, pyarrow, tqdm, scikit-learn) is safe to pip-install.

import sys, subprocess

pkgs = ['pandas', 'pyarrow', 'tqdm', 'scikit-learn']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--user', '-q'] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    # --user can fail in some venvs; retry without it
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

import importlib, site
importlib.invalidate_caches()

import pandas, pyarrow, tqdm, sklearn
print(f'pandas     {pandas.__version__}')
print(f'pyarrow    {pyarrow.__version__}')
print(f'tqdm       {tqdm.__version__}')
print(f'scikit     {sklearn.__version__}')
print('✓ All packages ready')

# ── If PyTorch was missing in cell 2, run this in a terminal: ────────────────
# nvidia-smi                              ← get your CUDA version
# pip install --user torch \              ← CUDA 12.1 example
#     --index-url https://download.pytorch.org/whl/cu121
# Then restart the kernel and run all cells again.

In [ ]:
# ── 4. Paths & sys.path ───────────────────────────────────────────────────────
import os, sys

WORKDIR    = os.path.expanduser('~/ai-coach')
DATA_FILE  = os.path.join(WORKDIR, 'ml', 'data', 'synthetic.parquet')
MODEL_FILE = os.path.join(WORKDIR, 'backend', 'models', 'cycling_coach.pt')
LAST_FILE  = os.path.join(WORKDIR, 'backend', 'models', 'cycling_coach_last.pt')

for p in [os.path.join(WORKDIR, 'backend'), WORKDIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ['PYTHONPATH'] = os.path.join(WORKDIR, 'backend')

os.makedirs(os.path.join(WORKDIR, 'ml', 'data'),        exist_ok=True)
os.makedirs(os.path.join(WORKDIR, 'backend', 'models'), exist_ok=True)

print(f'WORKDIR    : {WORKDIR}')
print(f'DATA_FILE  : {DATA_FILE}')
print(f'MODEL_FILE : {MODEL_FILE}')
print(f'LAST_FILE  : {LAST_FILE}')


In [ ]:
# ── 5. Generate / load training data ─────────────────────────────────────────
import os, sys, subprocess, multiprocessing, pandas as pd

WORKDIR   = os.path.expanduser('~/ai-coach')
DATA_FILE = os.path.join(WORKDIR, 'ml', 'data', 'synthetic.parquet')

needs_generate = True
if os.path.exists(DATA_FILE):
    try:
        _snap = pd.read_parquet(DATA_FILE, columns=['athlete_id', 'pc_5s_wkg'])
        _existing = _snap.athlete_id.nunique()
        del _snap
        if _existing >= ATHLETES:
            print(f'✓ Found existing data: {_existing:,} athletes — skipping generation')
            needs_generate = False
        else:
            print(f'⚠  Existing data has {_existing:,} athletes but need {ATHLETES:,} — regenerating…')
            os.remove(DATA_FILE)
    except Exception:
        print('⚠  Existing parquet is outdated (missing pc_5s_wkg) — regenerating…')
        os.remove(DATA_FILE)

if needs_generate:
    workers = max(1, multiprocessing.cpu_count() - 2)
    print(f'Generating {ATHLETES:,} athletes with {workers} workers…')
    subprocess.run([
        sys.executable, '-m', 'ml.training.generate_synthetic',
        '--athletes', str(ATHLETES),
        '--workers',  str(workers),
        '--output',   DATA_FILE,
    ], cwd=WORKDIR, check=True)

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — delete it and re-run this cell'
assert 'risk_inj_target' in df.columns, 'Old parquet — delete it and re-run this cell'
assert 'pc_5s_wkg'       in df.columns, 'Old parquet — delete it and re-run this cell'
print(f'✓ {len(df):,} rows | {df.athlete_id.nunique():,} athletes | {len(df.columns)} columns')
del df


In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
import os, sys, argparse, torch

WORKDIR    = os.path.expanduser('~/ai-coach')
DATA_FILE  = os.path.join(WORKDIR, 'ml', 'data', 'synthetic.parquet')
MODEL_FILE = os.path.join(WORKDIR, 'backend', 'models', 'cycling_coach.pt')
LAST_FILE  = os.path.join(WORKDIR, 'backend', 'models', 'cycling_coach_last.pt')
for p in [os.path.join(WORKDIR, 'backend'), WORKDIR]:
    if p not in sys.path: sys.path.insert(0, p)

for _k in list(sys.modules.keys()):
    if 'training.train' in _k:
        del sys.modules[_k]
from ml.training.train import train as run_training

# Batch size auto-scaled to VRAM
if   vram_gb >= 45: BATCH_SIZE = 4096   # ← A6000 48 GB
elif vram_gb >= 38: BATCH_SIZE = 2048
elif vram_gb >= 20: BATCH_SIZE = 1024
elif vram_gb >= 12: BATCH_SIZE = 512
elif vram_gb >= 6:  BATCH_SIZE = 256
else:               BATCH_SIZE = 64

STEPS_PER_EPOCH = 3000
EPOCHS          = 50
# torch.compile causes graph breaks on causal mask detection (Tensor.item()
# inside TransformerEncoder._detect_is_causal_mask), making it slower and
# using more VRAM. Keep it off.
USE_COMPILE = False

# ── Checkpoint / resume logic ─────────────────────────────────────────────────
# Prefer _last.pt (actual last epoch) over best .pt (best val epoch).
# Set FORCE_FRESH=True to wipe both and start from epoch 1.
FORCE_FRESH = False

if FORCE_FRESH:
    for _f in [MODEL_FILE, LAST_FILE]:
        if os.path.exists(_f): os.remove(_f)
    print('*** FORCE_FRESH: deleted checkpoints, starting from epoch 1 ***')
    CHECKPOINT = None
else:
    if os.path.exists(LAST_FILE):
        CHECKPOINT = LAST_FILE
    elif os.path.exists(MODEL_FILE):
        CHECKPOINT = MODEL_FILE
    else:
        CHECKPOINT = None

    if CHECKPOINT:
        _meta = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
        _ep   = _meta.get('metrics', {}).get('epoch', '?') if isinstance(_meta, dict) else '?'
        print(f'*** Resuming from {os.path.basename(CHECKPOINT)}: epoch {_ep} → will train epochs {_ep+1 if isinstance(_ep,int) else "?"}-{EPOCHS} ***')
    else:
        print('*** No checkpoint found — starting fresh ***')

print(f'GPU          : {torch.cuda.get_device_name(0) if device=="cuda" else device}')
print(f'VRAM         : {vram_gb} GB  →  batch: {BATCH_SIZE}')
print(f'Steps/epoch  : {STEPS_PER_EPOCH}  |  Epochs: {EPOCHS}')
print('─' * 60)

args = argparse.Namespace(
    data              = DATA_FILE,
    output            = MODEL_FILE,
    checkpoint        = CHECKPOINT,
    epochs            = EPOCHS,
    batch_size        = BATCH_SIZE,
    steps_per_epoch   = STEPS_PER_EPOCH,
    lr                = 3e-4,
    seq_len           = 90,
    val_frac          = 0.1,
    seed              = 42,
    d_model           = 256,
    nhead             = 8,
    num_layers        = 8,
    d_ff              = 1024,
    dropout           = 0.1,
    fast              = False,
    patience          = 20,
    compile           = USE_COMPILE,
    no_amp            = (device == 'cpu'),
    mask_workout_type = False,  # synthetic data: workout_type is ground-truth, keep it
)

run_training(args)
print(f'\n✓ Training complete!')
print(f'  Best model  → {MODEL_FILE}')
print(f'  Last epoch  → {LAST_FILE}')


In [ ]:
# ── 7. Sanity check ───────────────────────────────────────────────────────────
import os, sys, torch

WORKDIR    = os.path.expanduser('~/ai-coach')
MODEL_FILE = os.path.join(WORKDIR, 'backend', 'models', 'cycling_coach.pt')
for p in [os.path.join(WORKDIR, 'backend'), WORKDIR]:
    if p not in sys.path: sys.path.insert(0, p)

from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model         = cfg.get('d_model', 256),
    nhead           = cfg.get('nhead', 8),
    num_layers      = cfg.get('num_layers', 8),
    dim_feedforward = cfg.get('dim_feedforward', 1024),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()

metrics = ckpt.get('metrics', {})
print('Model loaded OK')
print(f'Parameters : {sum(p.numel() for p in m.parameters()):,}')
print(f'Best epoch : {metrics.get("epoch",    "n/a")}')
print(f'Val loss   : {metrics.get("val_loss", "n/a")}')
print(f'wt_acc     : {metrics.get("wt_acc",   "n/a")} %')
print(f'FTPΔ MAE   : {metrics.get("ftp_mae",  "n/a")} W')

In [ ]:
# ── 8. (Optional) Fine-tune on GoldenCheetah real data ───────────────────────
# If goldencheetah.parquet already exists, set FINETUNE=True and run — no CSV needed.
# If you need to convert from raw CSV, also set GC_INPUT_DIR to the CSV folder.

FINETUNE        = False
GC_INPUT_DIR    = os.path.expanduser('~/gc_data')   # ← only needed if parquet doesn't exist yet
GC_PARQUET      = os.path.join(WORKDIR, 'ml', 'data', 'goldencheetah.parquet')
FINETUNE_EPOCHS = 15
FINETUNE_LR     = 5e-5   # 10× lower than pre-training

if FINETUNE:
    import subprocess

    # ── Convert from raw CSV only if parquet doesn't exist yet ──────────────
    if not os.path.exists(GC_PARQUET):
        if not os.path.isdir(GC_INPUT_DIR):
            raise FileNotFoundError(
                f'goldencheetah.parquet not found at {GC_PARQUET}\n'
                f'and GC_INPUT_DIR not found at {GC_INPUT_DIR}\n'
                f'Either upload goldencheetah.parquet to ml/data/ or set GC_INPUT_DIR.'
            )
        print('Converting GoldenCheetah data…')
        env = os.environ.copy()
        env['PYTHONPATH'] = f"{os.path.join(WORKDIR, 'backend')}:{WORKDIR}"
        result = subprocess.run(
            [sys.executable, '-m', 'ml.training.convert_goldencheetah',
             '--input',  GC_INPUT_DIR,
             '--output', GC_PARQUET],
            cwd=WORKDIR, env=env, capture_output=False,
        )
        if result.returncode != 0:
            raise RuntimeError('convert_goldencheetah failed — see output above')
    else:
        print(f'✓ Using existing {GC_PARQUET}')

    import pandas as pd
    gc_df = pd.read_parquet(GC_PARQUET)
    print(f'GoldenCheetah: {len(gc_df):,} rows, {gc_df["athlete_id"].nunique()} athletes')

    print('\nFine-tuning on real data…')
    for _k in list(sys.modules.keys()):
        if 'training.train' in _k:
            del sys.modules[_k]
    from ml.training.train import train as run_training

    ft_args = argparse.Namespace(
        data              = GC_PARQUET,
        output            = MODEL_FILE,
        checkpoint        = MODEL_FILE,   # start from pre-trained weights
        epochs            = FINETUNE_EPOCHS,
        batch_size        = BATCH_SIZE,
        steps_per_epoch   = None,
        lr                = FINETUNE_LR,
        seq_len           = 90,
        val_frac          = 0.15,
        seed              = 42,
        d_model           = 256,
        nhead             = 8,
        num_layers        = 8,
        d_ff              = 1024,
        dropout           = 0.1,
        fast              = False,
        patience          = 8,
        compile           = False,
        no_amp            = (device == 'cpu'),
        mask_workout_type = True,   # real data: workout_type is inferred, zero it out
    )
    run_training(ft_args)
    print(f'✓ Fine-tuning complete! Model → {MODEL_FILE}')
else:
    print('Fine-tuning skipped (FINETUNE=False).')
    if os.path.exists(GC_PARQUET):
        print(f'  goldencheetah.parquet found — set FINETUNE=True to run.')
    else:
        print(f'  No goldencheetah.parquet found — upload it to ml/data/ first.')
